# Academic Indicator Dropout Model

This notebook trains the Flask app model using the same five inputs shown in the app:

- Attendance
- Assignments
- Marks
- Study Hrs
- Fees Up To Date (whether the student's tuition payments are current; 1 = Yes, 0 = No)

Attendance, Assignments, Marks, and Study Hrs are engineered from semester curricular-unit performance columns; Fees Up To Date comes directly from the tuition-status column.

In [ ]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = ROOT / "data" / "Predict Student Dropout and Academic Success.csv"
MODELS_DIR = ROOT / "models"
RESULTS_DIR = ROOT / "results"
MODEL_PATH = MODELS_DIR / "academic_indicator_model.pkl"
METRICS_PATH = RESULTS_DIR / "academic_indicator_metrics.txt"

FEATURE_NAMES = ["Attendance", "Assignments", "Marks", "Study Hrs", "Fees Up To Date"]

In [ ]:
def safe_divide(numerator, denominator):
    return np.divide(
        numerator,
        denominator,
        out=np.zeros_like(numerator, dtype=float),
        where=np.asarray(denominator) != 0,
    )


def build_academic_indicators(df):
    cu1_enrolled = df["Curricular units 1st sem (enrolled)"].astype(float)
    cu2_enrolled = df["Curricular units 2nd sem (enrolled)"].astype(float)
    cu1_approved = df["Curricular units 1st sem (approved)"].astype(float)
    cu2_approved = df["Curricular units 2nd sem (approved)"].astype(float)
    cu1_evaluations = df["Curricular units 1st sem (evaluations)"].astype(float)
    cu2_evaluations = df["Curricular units 2nd sem (evaluations)"].astype(float)
    cu1_grade = df["Curricular units 1st sem (grade)"].astype(float)
    cu2_grade = df["Curricular units 2nd sem (grade)"].astype(float)

    total_enrolled = cu1_enrolled + cu2_enrolled
    total_approved = cu1_approved + cu2_approved
    total_evaluations = cu1_evaluations + cu2_evaluations
    weighted_grade = safe_divide(
        (cu1_grade * cu1_enrolled) + (cu2_grade * cu2_enrolled),
        total_enrolled,
    )

    return pd.DataFrame(
        {
            "Attendance": np.clip(safe_divide(total_approved, total_enrolled) * 100, 0, 100),
            "Assignments": np.clip(safe_divide(total_evaluations, total_enrolled * 3) * 100, 0, 100),
            "Marks": np.clip(weighted_grade * 5, 0, 100),
            "Study Hrs": np.clip(total_enrolled, 0, 20),
            "Fees Up To Date": df["Tuition fees up to date"].astype(float),
        }
    )

In [ ]:
df = pd.read_csv(DATA_PATH)
X = build_academic_indicators(df)
y = (df["Target"] == "Dropout").astype(int)

X.describe().round(2)

In [ ]:
candidates = {
    "Logistic Regression": Pipeline(
        [
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(max_iter=5000, class_weight="balanced", random_state=42)),
        ]
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=500,
        min_samples_leaf=2,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
    ),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
}

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

results = []
for name, model in candidates.items():
    cv_auc = cross_val_score(model, X, y, cv=cv, scoring="roc_auc", n_jobs=-1)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    results.append(
        {
            "name": name,
            "model": model,
            "accuracy": accuracy_score(y_test, y_pred),
            "roc_auc": roc_auc_score(y_test, y_proba),
            "cv_auc_mean": cv_auc.mean(),
            "cv_auc_std": cv_auc.std(),
            "report": classification_report(y_test, y_pred),
            "confusion_matrix": confusion_matrix(y_test, y_pred),
        }
    )

pd.DataFrame(results).drop(columns=["model", "report", "confusion_matrix"]).sort_values("roc_auc", ascending=False)

In [ ]:
best = max(results, key=lambda row: row["roc_auc"])

model_bundle = {
    "model": best["model"],
    "model_name": best["name"],
    "feature_names": FEATURE_NAMES,
    "metrics": {
        "accuracy": round(best["accuracy"], 4),
        "roc_auc": round(best["roc_auc"], 4),
        "cv_roc_auc_mean": round(float(best["cv_auc_mean"]), 4),
        "cv_roc_auc_std": round(float(best["cv_auc_std"]), 4),
        "features": len(FEATURE_NAMES),
    },
}

MODELS_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)
joblib.dump(model_bundle, MODEL_PATH)

metrics_text = "\n".join(
    [
        "Academic Indicator Dropout Model Metrics",
        "=" * 50,
        f"Selected model       : {best['name']}",
        f"Input features       : {len(FEATURE_NAMES)}",
        f"Test accuracy        : {best['accuracy']:.6f}",
        f"Test ROC-AUC         : {best['roc_auc']:.6f}",
        f"5-fold CV ROC-AUC    : {best['cv_auc_mean']:.6f} +/- {best['cv_auc_std']:.6f}",
        "",
        "Features:",
        *[f"- {name}" for name in FEATURE_NAMES],
        "",
        "Classification Report:",
        best["report"],
        "Confusion Matrix:",
        str(best["confusion_matrix"]),
    ]
)
METRICS_PATH.write_text(metrics_text, encoding="utf-8")
print(metrics_text)
print(f"Saved model: {MODEL_PATH}")